# 📅 Exercício Extra 3: O Robô "Guarda-Costas" Autónomo 👤🛡️

Neste exercício, vamos usar um modelo de Inteligência Artificial chamado **Haar Cascade** para detetar rostos humanos em tempo real. 

Mas não nos vamos ficar pela imagem! O teu objetivo hoje é interligar a Inteligência Artificial com os motores do JetRacer para criar um comportamento autónomo: **o robô deve seguir-te pela sala**, mantendo sempre uma distância de segurança.

### 📏 Como medimos a distância com uma câmara normal?
O código mede a `largura` (em píxeis) do quadrado azul que envolve o teu rosto:
* Se a `largura > 130`: O teu rosto está grande no ecrã, o que significa que estás **muito perto**.
* Se a `largura < 80`: O teu rosto está pequeno no ecrã, o que significa que estás **muito longe**.
* Se estiver entre `80` e `130`: Estás na **distância ideal**.

---

### 🎯 O Teu Desafio de Programação
Olha para o código abaixo e descobre a zona marcada com `🎯 DESAFIO DOS ALUNOS`. Deves remover os comandos `pass` e escrever as regras de movimento usando as funções `mover(velocidade, direcao)` e `parar()` que aprendeste no Dia 1.

### 🛠️ Instruções Passo a Passo

1. ⚠️ **Segurança:** Confirma que o JetRacer está com as **rodas no ar** (em cima do bloco de suporte) para não fugir da mesa quando detetar o teu rosto!
2. Completa o código na zona do desafio.
3. Executa a célula (`Shift + Enter`).
4. Olha para a câmara. Aproxima-te e afasta-te do robô e repara se as rodas começam a girar para a frente ou para trás para tentar "seguir-te".
5. Quando terminares, clica no botão **❌ DESLIGAR SISTEMA**.

In [1]:
import cv2
import numpy as np
import ipywidgets as widgets
from IPython.display import display
from jetcam.csi_camera import CSICamera
from jethelper import mover, parar
import time

print("--- SISTEMA GUARDA-COSTAS AUTÓNOMO ATIVO ---")

# 1. Carregar o modelo de IA com o caminho padrão da Jetson Nano
caminho_xml = '/usr/share/opencv4/haarcascades/haarcascade_frontalface_default.xml'
classificador_rosto = cv2.CascadeClassifier(caminho_xml)

if classificador_rosto.empty():
    print("❌ Erro: Ficheiro IA não encontrado! Tenta o caminho sem o número 4.")

# 2. Inicializar a câmara e criar a interface
camera = CSICamera(width=300, height=300, capture_width=1280, capture_height=720, capture_fps=15)
imagem_widget = widgets.Image(format='jpeg', width=300, height=300)
botao_desligar = widgets.Button(description="❌ DESLIGAR SISTEMA", button_style='danger')

display(imagem_widget, botao_desligar)
sistema_ativo = True

# 3. Função de Inteligência e Controlo dos Motores
def processar_guarda_costas(change):
    global sistema_ativo
    if not sistema_ativo:
        parar()
        return
        
    frame = change['new']
    cinzento = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    cinzento = cv2.equalizeHist(cinzento) # Melhora o contraste para a IA
    
    # Detetar rostos
    rostos = classificador_rosto.detectMultiScale(cinzento, scaleFactor=1.05, minNeighbors=3, minSize=(40, 40))
    
    if len(rostos) == 0:
        print("A procurar alvo... 🔍                                 ", end='\r')
        parar() # Se perder o rosto de vista, o robô pára por segurança
    else:
        print(f"✅ Alvo na Mira! Rostos detetados: {len(rostos)}               ", end='\r')
        
    for (x, y, largura, altura) in rostos:
        # Desenha o quadrado azul no ecrã
        cv2.rectangle(frame, (x, y), (x + largura, y + altura), (255, 0, 0), 3)
        
        # --- 🎯 DESAFIO DOS ALUNOS: LÓGICA DOS MOTORES ---
        if largura > 130:
            # CASO A: O utilizador está muito perto!
            cv2.putText(frame, "Muito Perto! Recuar...", (x, y-10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 2)
            # [Escreve aqui o comando para o carro fazer MARCHA-ATRÁS suave (ex: velocidade -0.12)]
            mover(-0.12, 0.0)
            
        elif largura < 80:
            # CASO B: O utilizador afastou-se e está longe!
            cv2.putText(frame, "Muito Longe! Avançar...", (x, y-10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)
            # [Escreve aqui o comando para o carro AVANÇAR para a frente suavemente (ex: velocidade 0.12)]
            mover(0.12, 0.0)
            
        else:
            # CASO C: A distância está perfeita.
            cv2.putText(frame, "Distância Correta. Imobilizado.", (x, y-10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 0), 2)
            # [Escreve aqui o comando para PARAR o carro]
            parar()
            
    # Atualiza o widget com o vídeo processado
    _, jpeg = cv2.imencode('.jpg', frame)
    imagem_widget.value = jpeg.tobytes()
    time.sleep(0.02)

# Ativa a monitorização da câmara
camera.observe(processar_guarda_costas, names='value')

# 4. Botão de Paragem de Emergência
def desligar_tudo(b):
    global sistema_ativo
    sistema_ativo = False
    print("\n[A desligar motores e a trancar câmara...]")
    parar()
    try:
        camera.unobserve(processar_guarda_costas, names='value')
        camera.running = False
    except:
        pass
    botao_desligar.description = "🛑 SISTEMA INATIVO"
    botao_desligar.button_style = "info"
    botao_desligar.disabled = True
    print("Plataforma imobilizada com sucesso!")

botao_desligar.on_click(desligar_tudo)
camera.running = True

WARNNIG: Jetson.GPIO library has not been verified with this carrier board,


--- SISTEMA GUARDA-COSTAS AUTÓNOMO ATIVO ---


Image(value=b'', format='jpeg', height='300', width='300')

Button(button_style='danger', description='❌ DESLIGAR SISTEMA', style=ButtonStyle())